In [1]:
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
)
import shap

In [2]:
RANDOM_STATE = 42

def select_features(X_train: pd.DataFrame, y_train: pd.Series) -> list[str]:
    print("\nRunning RFECV feature selection …")
    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X_train)
    rf = RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    sel = RFECV(
        rf, step=1, cv=5, scoring="roc_auc",
        min_features_to_select=min(MIN_FEATURES, X_train.shape[1]),
        n_jobs=-1,
    )
    sel.fit(X_imp, y_train)
    chosen = list(X_train.columns[sel.support_])
    print(f"  Selected {len(chosen)} / {X_train.shape[1]} features:")
    print("  ", ", ".join(chosen))
    return chosen

def make_booster():
    """XGBoost if installed, otherwise sklearn HistGradientBoosting."""
    return XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=RANDOM_STATE,
    ), "XGBoost"

def evaluate(name, model, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    print(f"\n===== {name} =====")
    print(f"  AUROC : {roc_auc_score(y_test, proba):.3f}")
    print(f"  AUPRC : {average_precision_score(y_test, proba):.3f}")
    print("  Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))

def compute_shap_importance(pipeline, features, X_test):
    X_imp = pipeline.named_steps["impute"].transform(X_test)
    explainer = shap.TreeExplainer(pipeline.named_steps["clf"])
    vals = explainer.shap_values(X_imp)
    if isinstance(vals, list):
        vals = vals[1]
    imp = (pd.Series(np.abs(vals).mean(axis=0), index=features)
           .sort_values(ascending=False))
    print("\nMean |SHAP| feature importance:")
    print(imp.to_string())
    return imp

In [3]:
store = pd.HDFStore('data/store.h5', mode='r')
df = store['df']
X, y = store['X'], store['y']
store.close()

In [5]:
MIN_FEATURES = 5

print(f"  Features available : {X.shape[1]}")
print(f"  CVD positive       : {y.mean():.1%}  ({y.sum()} / {len(y)})")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)

selected = select_features(X_train, y_train)
X_train, X_test = X_train[selected], X_test[selected]

sw = compute_sample_weight("balanced", y_train)

# Logistic regression baseline
lr = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, class_weight="balanced")),
])
lr.fit(X_train, y_train)
evaluate("Logistic Regression (baseline)", lr, X_test, y_test)

# Primary gradient-boosted model
booster, bname = make_booster()
boost = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf",    booster),
])
boost.fit(X_train, y_train, clf__sample_weight=sw)
evaluate(f"{bname} (primary)", boost, X_test, y_test)

shap_imp = compute_shap_importance(boost, selected, X_test)

bundle = {
    "model":             boost,
    "lr_model":          lr,
    "features":          selected,
    "feature_meta":      {
        "age":          {"label": "Age",                           "unit": "years"},
        "female":       {"label": "Sex (1=female, 0=male)",        "unit": "binary"},
        "race":         {"label": "Race/ethnicity code",           "unit": "category"},
        "education":    {"label": "Education level (1–5)",         "unit": "ordinal"},
        "income":       {"label": "Income-to-poverty ratio",       "unit": "ratio"},
        "sbp":          {"label": "Systolic blood pressure",       "unit": "mmHg"},
        "dbp":          {"label": "Diastolic blood pressure",      "unit": "mmHg"},
        "pulse":        {"label": "Resting pulse",                 "unit": "bpm"},
        "bmi":          {"label": "Body mass index",               "unit": "kg/m²"},
        "waist":        {"label": "Waist circumference",           "unit": "cm"},
        "total_chol":   {"label": "Total cholesterol",             "unit": "mg/dL"},
        "hdl":          {"label": "HDL cholesterol",               "unit": "mg/dL"},
        "ldl":          {"label": "LDL cholesterol",               "unit": "mg/dL"},
        "trig":         {"label": "Triglycerides",                 "unit": "mg/dL"},
        "non_hdl":      {"label": "Non-HDL cholesterol",           "unit": "mg/dL"},
        "hba1c":        {"label": "Glycohemoglobin (HbA1c)",       "unit": "%"},
        "glucose":      {"label": "Fasting glucose",               "unit": "mg/dL"},
        "insulin":      {"label": "Fasting insulin",               "unit": "µU/mL"},
        "crp":          {"label": "High-sensitivity CRP",          "unit": "mg/L"},
        "creatinine":   {"label": "Serum creatinine",              "unit": "mg/dL"},
        "bun":          {"label": "Blood urea nitrogen",           "unit": "mg/dL"},
        "uric_acid":    {"label": "Uric acid",                     "unit": "mg/dL"},
        "uacr":         {"label": "Urine albumin-creatinine ratio","unit": "mg/g"},
        "wbc":          {"label": "White blood cell count",        "unit": "1000/µL"},
        "diabetes_dx":  {"label": "Diabetes diagnosis (1=yes)",    "unit": "binary"},
        "htn_dx":       {"label": "Hypertension diagnosis (1=yes)","unit": "binary"},
        "highchol_dx":  {"label": "High cholesterol dx (1=yes)",   "unit": "binary"},
        "family_hx":    {"label": "Family history of CVD (1=yes)", "unit": "binary"},
        "kidney_dx":    {"label": "Kidney disease dx (1=yes)",     "unit": "binary"},
        "smoking":      {"label": "Smoking (0=never,1=former,2=now)","unit": "ordinal"},
        "alcohol_day":  {"label": "Avg drinks per day",            "unit": "drinks"},
        "sedentary_min":{"label": "Sedentary time",                "unit": "min/day"},
        "sleep_hours":  {"label": "Sleep duration",                "unit": "hours"},
    },
    "shap_importance":   shap_imp.to_dict() if shap_imp is not None else None,
    "label":             "Prevalent CVD (heart attack or stroke ever diagnosed)",
    "data_source":       "NHANES 2017-2018",
    "model_type":        bname,
    "train_positives":   int(y_train.sum()),
    "train_total":       int(len(y_train)),
}

joblib.dump(bundle, "./cvd_model_bundle.joblib")
print(f"  Primary model  : {bname}")
print(f"  Selected feats : {selected}")

  Features available : 32
  CVD positive       : 11.8%  (460 / 3882)

Running RFECV feature selection …
  Selected 29 / 32 features:
   age, race, education, income, sbp, dbp, bmi, waist, total_chol, hdl, ldl, trig, non_hdl, hba1c, glucose, insulin, crp, creatinine, bun, uric_acid, uacr, wbc, diabetes_dx, htn_dx, highchol_dx, family_hx, smoking, sedentary_min, sleep_hours

===== Logistic Regression (baseline) =====
  AUROC : 0.788
  AUPRC : 0.314
  Confusion matrix:
 [[629 227]
 [ 36  79]]
              precision    recall  f1-score   support

           0      0.946     0.735     0.827       856
           1      0.258     0.687     0.375       115

    accuracy                          0.729       971
   macro avg      0.602     0.711     0.601       971
weighted avg      0.864     0.729     0.774       971


===== XGBoost (primary) =====
  AUROC : 0.785
  AUPRC : 0.280
  Confusion matrix:
 [[762  94]
 [ 67  48]]
              precision    recall  f1-score   support

           0    